### SMOTENC
SMOTENC is used when the dataset contains both numerical and categorical features.

For numerical features, it works like regular SMOTE. For categorical features, it selects category values from the nearest neighbors instead of creating new numeric-style values.

In [40]:
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTENC

In [ ]:
# Load the Adult dataset from OpenML.
adult = fetch_openml("adult", version=2, as_frame=True)
X = adult.data
y = adult.target

y.value_counts()
# The class counts show that the target variable is imbalanced.
# SMOTENC will be used later to balance the training data.

class
<=50K    37155
>50K     11687
Name: count, dtype: int64

In [16]:
# Identify categorical and numerical columns in the dataset.
all_categorical_columns = X.select_dtypes(include=["object", "category"]).columns
all_numerical_columns = X.select_dtypes(include=["float64", "int64"]).columns
print(f"Categorical columns: {all_categorical_columns}")
print(f"Numerical columns: {all_numerical_columns}")

Categorical columns: Index(['workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'sex', 'native-country'],
      dtype='str')
Numerical columns: Index(['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='str')


In [36]:
# Split the dataset into training and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [37]:
# Create a clean copy of the training data before handling missing values.
X_train_clean = X_train.copy()

# Create imputers with suitable strategies for each data type.
numerical_imputer = SimpleImputer(strategy="median")  # average or median
categorical_imputer = SimpleImputer(strategy="most_frequent")  # mode

In [38]:
X_train_clean[all_categorical_columns] = categorical_imputer.fit_transform(
    X_train_clean[all_categorical_columns]
)
X_train_clean[all_numerical_columns] = numerical_imputer.fit_transform(
    X_train_clean[all_numerical_columns]
)

In [47]:
y_train.value_counts()

class
<=50K    29724
>50K      9349
Name: count, dtype: int64

In [ ]:
# Apply SMOTENC to create a balanced training dataset.
# SMOTENC needs the index positions of categorical columns to handle them correctly.
categorical_index = [X_test.columns.get_loc(col) for col in all_categorical_columns]

smotenc = SMOTENC(
    categorical_features=categorical_index,
    sampling_strategy="auto",
    random_state=42,
    k_neighbors=5,  # Number of nearest neighbors used while creating synthetic samples.
)

X_train_smotenc, y_train_smotenc = smotenc.fit_resample(X_train_clean, y_train)

# Check the class distribution after resampling.
y_train_smotenc.value_counts()

class
<=50K    29724
>50K     29724
Name: count, dtype: int64